# Simulation remote control

The simulation-side counterpart to `uniwheel_rc/` - drive the robot with arrow keys in
the MuJoCo viewer, but through the actual trained stage-05 policy (not raw motor
control like stage 01's `test_motion.ipynb`). This is the same thing the real
hardware's `uniwheel_rc.ino` does when you drag the phone-browser slider - here, the
slider is the Up/Down arrow keys, and the "phone" is this notebook.

- **Up / Down arrows**: raise/lower `cmd_vel` (normalized [-1, 1], step 0.1)
- **Backspace**: reset the sim (MuJoCo's own binding) and zero `cmd_vel`

Same confirmed limitation as everywhere else this policy is used: `cmd_vel` is a
**direction** control, not a precise speed dial - see
`../05-command-conditioning/README.md`'s velocity-tracking section.


In [1]:
# Import standard libraries
import math
from pathlib import Path
import sys
import time

# Third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer
import numpy as np
import torch

sys.path.append("..")
from rl.ppo_trainer import PPOConfig, Agent


In [2]:
# Settings
MJCF_PATH = Path("../../mechanical/urdf/uniwheel/uniwheel.xml")

# Stage 05's verified checkpoint (see ../05-command-conditioning/README.md)
CKPT_PATH = Path(
    "../05-command-conditioning/runs/"
    "UniWheel-v0__uniwheel-ppo-cmd-phase-10__42__1788754064/best_model.cleanrl_model"
)

CMD_VEL_STEP = 0.1     # How much Up/Down changes cmd_vel by, per press
ALPHA = 0.98           # Complementary filter weight - MUST match uniwheel_env_cmd.py
TIP_THRESHOLD_DEG = 30.0
PRINT_EVERY = 50

# Actuator/sensor names (from the MJCF file)
WHEEL_MOTOR = "wheel_motor"
REACTION_WHEEL_MOTOR = "reaction_wheel_motor"
IMU_ACCEL = "imu_accel"
IMU_GYRO = "imu_gyro"
WHEEL_VEL = "wheel_vel"
REACTION_WHEEL_VEL = "reaction_wheel_vel"

# Keys
KEY_BACKSPACE = 259
KEY_UP = 265
KEY_DOWN = 264


In [3]:
# Global: current velocity command, adjusted by key_callback
cmd_vel = {"value": 0.0}

def clamp(x):
    return max(-1.0, min(1.0, x))

def key_callback(keycode):
    """Handle a key press"""
    if keycode == KEY_BACKSPACE:
        cmd_vel["value"] = 0.0
        return
    if keycode == KEY_UP:
        cmd_vel["value"] = clamp(cmd_vel["value"] + CMD_VEL_STEP)
    elif keycode == KEY_DOWN:
        cmd_vel["value"] = clamp(cmd_vel["value"] - CMD_VEL_STEP)


In [4]:
# Load the MuJoCo model
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))
data = mujoco.MjData(model)

wheel_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, WHEEL_MOTOR)
reaction_wheel_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, REACTION_WHEEL_MOTOR)
print(f"Wheel motor ID: {wheel_motor_id}")
print(f"Reaction wheel motor ID: {reaction_wheel_motor_id}")


Wheel motor ID: 0
Reaction wheel motor ID: 1


In [5]:
# Load the trained policy (stage 05 - 7 observation inputs, includes cmd_vel)
class _Spaces:
    pass
dummy_envs = _Spaces()
dummy_envs.single_observation_space = type("S", (), {"shape": (7,)})()
dummy_envs.single_action_space = type("S", (), {"shape": (2,)})()

ppo_config = PPOConfig(actor_hidden_layers=2, actor_hidden_size=48,
                        critic_hidden_layers=2, critic_hidden_size=48)
agent = Agent(dummy_envs, ppo_config)
agent.load_state_dict(torch.load(CKPT_PATH, map_location="cpu", weights_only=True))
agent.eval()
print(f"Loaded policy from {CKPT_PATH}")


Loaded policy from ../05-command-conditioning/runs/UniWheel-v0__uniwheel-ppo-cmd-phase-10__42__1788754064/best_model.cleanrl_model


In [6]:
# Resets simulation data to defaults
mujoco.mj_resetData(model, data)

# Complementary-filter state
pitch = 0.0
roll = 0.0
prev_time = 0.0
steps = 0

with mujoco.viewer.launch_passive(model, data, key_callback=key_callback) as viewer:
    # Define free-look camera (control with mouse)
    viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    viewer.cam.lookat[:] = [0, 0, 0.08]
    viewer.cam.distance = 0.8
    viewer.cam.azimuth = 45
    viewer.cam.elevation = -25

    while viewer.is_running():
        step_start = time.time()

        # Detect MuJoCo reset (backspace) and reset filter state too
        if data.time < prev_time:
            pitch = 0.0
            roll = 0.0
            mujoco.mj_forward(model, data)
        prev_time = data.time

        # Read sensors - same complementary filter as uniwheel_env_cmd.py's _get_obs()
        accel_x, accel_y, accel_z = data.sensor(IMU_ACCEL).data
        pitch_rate = data.sensor(IMU_GYRO).data[1]
        roll_rate = data.sensor(IMU_GYRO).data[0]

        accel_pitch = -1 * math.atan2(accel_x, accel_z)
        accel_roll = math.atan2(accel_y, accel_z)
        pitch = ALPHA * (pitch + pitch_rate * model.opt.timestep) + (1 - ALPHA) * accel_pitch
        roll = ALPHA * (roll + roll_rate * model.opt.timestep) + (1 - ALPHA) * accel_roll

        wheel_vel = data.sensor(WHEEL_VEL).data[0]
        reaction_wheel_vel = data.sensor(REACTION_WHEEL_VEL).data[0]

        # Build observation and run the policy (deterministic action = actor_mean,
        # for a stable, repeatable demo rather than the stochastic training policy)
        obs = np.array([pitch, roll, pitch_rate, roll_rate, wheel_vel,
                         reaction_wheel_vel, cmd_vel["value"]], dtype=np.float32)
        with torch.no_grad():
            x = torch.as_tensor(obs).unsqueeze(0)
            action = agent.actor_mean(x).squeeze(0).numpy()
        action = np.clip(action, -1.0, 1.0)

        if abs(pitch) > math.radians(TIP_THRESHOLD_DEG) or abs(roll) > math.radians(TIP_THRESHOLD_DEG):
            data.ctrl[wheel_motor_id] = 0.0
            data.ctrl[reaction_wheel_motor_id] = 0.0
        else:
            data.ctrl[wheel_motor_id] = action[0]
            data.ctrl[reaction_wheel_motor_id] = action[1]

        mujoco.mj_step(model, data)
        viewer.sync()

        steps += 1
        if steps % PRINT_EVERY == 0:
            clear_output(wait=True)
            print(f"cmd_vel: {cmd_vel['value']:+.2f}   pitch: {math.degrees(pitch):6.2f} deg   "
                  f"roll: {math.degrees(roll):6.2f} deg")

        # Slow the simulation down so we can see the robot behave in real time
        slack = model.opt.timestep - (time.time() - step_start)
        if slack > 0:
            time.sleep(slack)


cmd_vel: +0.00   pitch:  -0.08 deg   roll:  -0.05 deg
